# Divisão dos dados entre treino e teste

Este notebook implementa a divisão dos dados definida no registro de decisão da estratégia de split e documentada na Seção 4.3.1. Ele não treina modelo algum: sua única saída é a partição, gravada em arquivo, que todos os experimentos passam a ler em vez de sortear novamente.

## O que a partição precisa garantir

Três exigências orientam a implementação.

A primeira é o agrupamento por configuração. As linhas não são independentes entre si, pois a base contém repetições da mesma configuração de fogão. Nenhuma configuração pode aparecer simultaneamente em treino e em teste, sob pena de a métrica deixar de responder à pergunta de interesse, que é o erro diante de uma configuração ainda não ensaiada.

A segunda é a validade simultânea para as duas versões da base. Encontram-se em avaliação uma versão em que a coluna de perdas mantém suas 13 categorias e outra em que ela é reduzida ao perfil de perdas. As duas produzem chaves de configuração diferentes, de 32 e de 25 grupos respectivamente. Como a chave do perfil é um agrupamento mais grosso da chave de 13 categorias, uma partição construída sobre a primeira respeita automaticamente a segunda, ao passo que o inverso não é verdadeiro. A partição é, por isso, construída sobre a chave do perfil, e a relação de agrupamento é verificada nas células seguintes em vez de assumida.

A terceira é a unicidade. A mesma partição é usada pelas sete áreas, pelas duas versões de tratamento das perdas e pelos dois tratamentos de valores ausentes. Sem isso, as comparações entre esses fatores ficariam confundidas com a variação do sorteio, cuja magnitude foi medida em 2,25 graus Celsius de desvio, superior à diferença esperada entre as alternativas comparadas.

## 1. Configuração

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S3/preparacao_testes"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import json
import os

import numpy as np
import pandas as pd

# Pastas das duas versoes em avaliacao
PASTA_PERFIL = "colabs_S3"
PASTA_13CAT = "colabs_S3"
ARQ_PERFIL = "dataset_mainback_perfil.csv"
ARQ_13CAT = "dataset_mainback_13cat.csv"

# Qualquer area serve como referencia: as colunas de configuracao sao as mesmas
# nos sete arquivos, e e so delas que a particao depende.

FRACAO_TESTE = 0.20   # proporcao de configuracoes reservadas ao teste
SEMENTE = 42          # fixada para que a particao seja reproduzivel

SAIDA_CSV = "particao_treino_teste.csv"
SAIDA_JSON = "particao_treino_teste.json"

# A chave da particao reune apenas as sete variaveis de configuracao da planilha,
# e deliberadamente nao inclui os atributos extraidos do motivo do ensaio, de
# prefixo cfg_. A razao e que uma particao construida sobre chave mais grossa
# respeita automaticamente qualquer chave mais fina obtida por subdivisao de seus
# grupos, de modo que esta escolha protege as duas versoes da base e sobrevive a
# revisoes do conjunto de entrada, sem exigir novo sorteio. Incluir cfg_ aqui
# elevaria a chave de 25 para 41 configuracoes, reduziria os grupos a um unico
# ensaio na maioria dos casos e faria a particao mudar a cada alteracao de feature.
PREFIXOS_CONFIG = ("blower_direction_", "housing_", "board_position_",
                   "run_case_", "losses_", "perfil_perdas_",
                   "deflector_hole_", "material_coil_support_")


# O notebook pode ser executado a partir da pasta notebooks, da raiz do repositorio
# ou do Colab, onde os arquivos costumam ficar soltos ao lado do notebook. A busca
# abaixo cobre esses casos e falha com mensagem explicita caso nao encontre.
BASES = [".", "..", os.path.join("..", ".."),
         "notebooks", os.path.join("..", "notebooks"),
         "preparacao_testes", os.path.join("..", "preparacao_testes"),
         os.path.join("notebooks", "preparacao_testes")]


def resolve(pasta, arquivo):
    """Localiza um arquivo de dados, com ou sem a pasta da versao."""
    tentativas = []
    # A pasta declarada tem prioridade sobre o arquivo solto: sem isso, um
    # homonimo numa pasta vizinha pode ser lido no lugar do arquivo pedido.
    for partes in ((pasta, arquivo), (arquivo,)):
        for base in BASES:
            caminho = os.path.join(base, *partes)
            tentativas.append(caminho)
            if os.path.exists(caminho):
                return caminho
    raise FileNotFoundError(
        "Arquivo nao encontrado: " + arquivo
        + ". Diretorio atual: " + os.path.abspath(".")
        + ". Caminhos tentados: " + "; ".join(tentativas)
    )


def colunas_de_configuracao(df):
    """Colunas que compoem o vetor de entrada visto pelo modelo."""
    return [c for c in df.columns if c.startswith(PREFIXOS_CONFIG)]


def chave(df):
    """Identificador textual da configuracao de cada ensaio."""
    return df[colunas_de_configuracao(df)].astype(str).agg("|".join, axis=1)


print("Diretorio atual:", os.path.abspath("."))
print("Semente:", SEMENTE, "| fracao de teste:", FRACAO_TESTE)

## 2. Leitura das duas versões e verificação dos pressupostos

Antes de sortear qualquer coisa, os três pressupostos do desenho são verificados sobre os dados. Caso algum falhe, a execução é interrompida, uma vez que a partição resultante seria inválida.

In [ ]:
caminho_perfil = resolve(PASTA_PERFIL, ARQ_PERFIL)
caminho_13cat = resolve(PASTA_13CAT, ARQ_13CAT)
print("Lendo:", caminho_perfil)
print("Lendo:", caminho_13cat)
print()

perfil = pd.read_csv(caminho_perfil)
cat13 = pd.read_csv(caminho_13cat)

k_perfil = chave(perfil)
k_13cat = chave(cat13)

print("Versao perfil:", perfil.shape, "->", k_perfil.nunique(), "configuracoes")
print("Versao 13cat: ", cat13.shape, "->", k_13cat.nunique(), "configuracoes")
print()

# Pressuposto 1: as duas versoes descrevem os mesmos ensaios, na mesma ordem
assert len(perfil) == len(cat13), "as duas versoes tem numero de linhas diferente"
# A comparacao usa apenas as seis features que nao envolvem as perdas: as colunas
# de perdas tem o mesmo prefixo nas duas versoes, mas significados diferentes,
# pois numa delas os rotulos ja foram agregados em perfis.
def sem_perdas(df):
    return [c for c in colunas_de_configuracao(df)
            if not c.startswith(("losses_", "perfil_perdas_"))]

comuns = [c for c in sem_perdas(perfil) if c in sem_perdas(cat13)]
alinhadas = bool((perfil[comuns].values == cat13[comuns].values).all())
print(f"Linhas alinhadas entre as versoes ({len(comuns)} colunas comuns):", alinhadas)
assert alinhadas, "as linhas das duas versoes nao correspondem aos mesmos ensaios"

# Pressuposto 2: a chave do perfil e um agrupamento mais grosso da chave de 13 categorias.
# Cada configuracao da versao detalhada deve estar contida em exatamente uma do perfil.
por_grupo = pd.DataFrame({"fina": k_13cat, "grossa": k_perfil}).groupby("fina")["grossa"].nunique()
print("Cada configuracao de 13cat cai em uma unica configuracao de perfil:", bool(por_grupo.max() == 1))
assert por_grupo.max() == 1, "a chave do perfil nao e um agrupamento da chave de 13 categorias"

# Pressuposto 3: nenhuma configuracao vazia
assert k_perfil.notna().all(), "ha ensaios sem configuracao definida"
print()
print("Pressupostos verificados. A particao sera construida sobre a chave do perfil.")

## 3. Estrutura dos grupos

A composição dos grupos é registrada antes do sorteio, pois é ela que justifica tanto a estratificação quanto a escolha do desenho de validação.

In [ ]:
ensaios = pd.DataFrame({"run_idx": np.arange(len(perfil)), "configuracao": k_perfil})

# Rotulo do tipo de ensaio, usado na estratificacao
cols_rc = [c for c in perfil.columns if c.startswith("run_case_")]
ensaios["run_case"] = perfil[cols_rc].idxmax(axis=1).str.replace("run_case_", "", regex=False)

tamanhos = ensaios["configuracao"].value_counts()
print(f"Ensaios: {len(ensaios)}")
print(f"Configuracoes distintas: {len(tamanhos)}")
print(f"Configuracoes repetidas: {int((tamanhos > 1).sum())}, "
      f"cobrindo {int(tamanhos[tamanhos > 1].sum())} ensaios")
print(f"Tamanho dos grupos: {sorted(tamanhos.tolist(), reverse=True)}")
print()

# Cada configuracao pertence a um unico tipo de ensaio
config = ensaios.drop_duplicates("configuracao")[["configuracao", "run_case"]]
assert ensaios.groupby("configuracao")["run_case"].nunique().max() == 1
print("Configuracoes por tipo de ensaio:")
print(config["run_case"].value_counts().to_string())

## 4. Sorteio do conjunto de teste

O conjunto de teste é sorteado por configuração e estratificado pelo tipo de ensaio, de modo que a composição do teste acompanhe, na medida do possível, a composição da base.

Duas restrições são impostas ao sorteio. Nenhum tipo de ensaio pode ter todas as suas configurações levadas ao teste, sob pena de o modelo ser avaliado por extrapolação pura naquele modo de operação. E tipos de ensaio com pouquíssimas configurações podem acabar sem representação no teste, situação que é registrada em vez de forçada: reservar ao teste uma das duas únicas configurações de broil c1 deixaria o treino com uma só, o que não sustentaria aprendizado algum.

In [ ]:
rng = np.random.default_rng(SEMENTE)

configs_teste = []
composicao = []

tamanho_config = ensaios["configuracao"].value_counts()

for rc, grupo in config.groupby("run_case"):
    disponiveis = grupo["configuracao"].tolist()
    rng.shuffle(disponiveis)

    # O alvo e proporcional em ENSAIOS, e nao em configuracoes. Como os grupos sao
    # muito desiguais, equilibrar apenas a contagem de configuracoes deixaria o
    # treino com pouquissimos ensaios de algum tipo.
    ensaios_do_tipo = int(tamanho_config.loc[disponiveis].sum())
    alvo = ensaios_do_tipo * FRACAO_TESTE

    escolhidas, acumulado = [], 0
    for c in disponiveis:
        if len(escolhidas) == len(disponiveis) - 1:
            break  # ao menos uma configuracao permanece no treino
        if acumulado + tamanho_config[c] <= alvo:
            escolhidas.append(c)
            acumulado += tamanho_config[c]

    configs_teste.extend(escolhidas)
    composicao.append({"run_case": rc,
                       "configuracoes": len(disponiveis),
                       "ensaios": ensaios_do_tipo,
                       "config_teste": len(escolhidas),
                       "ensaios_teste": acumulado,
                       "ensaios_treino": ensaios_do_tipo - acumulado})

configs_teste = set(configs_teste)
composicao = pd.DataFrame(composicao)

print("Composicao da particao por tipo de ensaio:")
print(composicao.to_string(index=False))
print()

sem_teste = composicao.loc[composicao["ensaios_teste"] == 0, "run_case"].tolist()
if sem_teste:
    print("Tipos de ensaio sem representacao no teste:", sem_teste)
    print("A limitacao decorre do numero de configuracoes disponiveis e deve constar da documentacao.")

## 5. Dobras de validação sobre o conjunto de treino

A validação é feita por leave-one-group-out sobre as configurações de treino: cada configuração é deixada de fora uma vez, e o modelo é treinado com as demais. A escolha foi justificada no registro de decisão pelo menor viés, uma vez que o modelo entregue é treinado com o conjunto de treino inteiro, e não com uma fração dele.

A composição de cada dobra é registrada, pois uma dobra cujo treino não contenha algum tipo de ensaio produz avaliação por extrapolação e precisa ser identificada na leitura dos resultados.

In [ ]:
ensaios["conjunto"] = np.where(ensaios["configuracao"].isin(configs_teste), "teste", "treino")

configs_treino = sorted(ensaios.loc[ensaios["conjunto"] == "treino", "configuracao"].unique())
print(f"Configuracoes em treino: {len(configs_treino)} | em teste: {len(configs_teste)}")
print(f"Ensaios em treino: {int((ensaios['conjunto'] == 'treino').sum())} | "
      f"em teste: {int((ensaios['conjunto'] == 'teste').sum())}")
print()

# Uma dobra por configuracao de treino
dobras = {c: i for i, c in enumerate(configs_treino)}
ensaios["dobra"] = ensaios["configuracao"].map(dobras)

rc_por_config = config.set_index("configuracao")["run_case"]
tipos = set(rc_por_config.loc[configs_treino].unique())

incompletas = []
for c, i in dobras.items():
    treino_da_dobra = [x for x in configs_treino if x != c]
    faltando = tipos - set(rc_por_config.loc[treino_da_dobra].unique())
    if faltando:
        incompletas.append({"dobra": i, "tipo_ausente": ", ".join(sorted(faltando))})

print(f"Dobras de validacao: {len(dobras)}")
if incompletas:
    print("Dobras cujo treino nao contem algum tipo de ensaio:")
    print(pd.DataFrame(incompletas).to_string(index=False))
    print("Os resultados dessas dobras devem ser lidos como extrapolacao.")
else:
    print("Todas as dobras treinam com todos os tipos de ensaio representados.")

## 6. Verificação da partição

As verificações abaixo são o teste programático de ausência de vazamento. Elas falham de forma explícita caso alguma das garantias do desenho seja violada.

In [ ]:
idx_treino = ensaios.loc[ensaios["conjunto"] == "treino", "run_idx"].tolist()
idx_teste = ensaios.loc[ensaios["conjunto"] == "teste", "run_idx"].tolist()

# 1. Nenhum ensaio em dois conjuntos, e nenhum ensaio de fora
assert set(idx_treino).isdisjoint(idx_teste)
assert len(idx_treino) + len(idx_teste) == len(ensaios)

# 2. Nenhuma configuracao nos dois conjuntos, sob a chave do perfil
c_tr = set(ensaios.loc[ensaios["conjunto"] == "treino", "configuracao"])
c_te = set(ensaios.loc[ensaios["conjunto"] == "teste", "configuracao"])
assert c_tr.isdisjoint(c_te)

# 3. Nenhuma configuracao nos dois conjuntos sob a chave detalhada de 13 categorias.
#    Esta e a verificacao que garante a validade da particao para a outra versao da base.
f_tr = set(k_13cat[idx_treino])
f_te = set(k_13cat[idx_teste])
assert f_tr.isdisjoint(f_te), "particao invalida para a versao de 13 categorias"

# 4. Todo tipo de ensaio presente no treino
assert set(ensaios.loc[ensaios["conjunto"] == "treino", "run_case"]) == set(ensaios["run_case"])

# 5. Cada configuracao de treino pertence a exatamente uma dobra
assert ensaios.loc[ensaios["conjunto"] == "treino", "dobra"].notna().all()
assert ensaios.loc[ensaios["conjunto"] == "teste", "dobra"].isna().all()

print("Verificacoes da particao")
print(f"  ensaios treino / teste                 : {len(idx_treino)} / {len(idx_teste)}")
print(f"  configuracoes treino / teste (perfil)  : {len(c_tr)} / {len(c_te)}")
print(f"  configuracoes treino / teste (13cat)   : {len(f_tr)} / {len(f_te)}")
print(f"  intersecao de configuracoes (perfil)   : {len(c_tr & c_te)}")
print(f"  intersecao de configuracoes (13cat)    : {len(f_tr & f_te)}")
print()
print("Composicao por tipo de ensaio:")
print(pd.crosstab(ensaios["run_case"], ensaios["conjunto"]).to_string())
print()
print("Todas as verificacoes passaram.")

## 7. Gravação da partição

A partição é gravada em dois formatos. O arquivo em CSV associa cada ensaio ao seu conjunto e à sua dobra, e é o que os notebooks de treino leem. O arquivo em JSON registra os parâmetros do sorteio e as listas de configurações, servindo de registro auditável de como a partição foi produzida.

Os notebooks de treino devem ler estes arquivos em vez de sortear novamente. A fixação da semente não basta por si só, uma vez que qualquer alteração na ordenação das linhas produziria sorteio distinto.

In [ ]:
particao = ensaios[["run_idx", "conjunto", "dobra", "run_case", "configuracao"]].copy()

registro = {
    "semente": SEMENTE,
    "fracao_teste": FRACAO_TESTE,
    "chave": "configuracao sob o perfil de perdas (agrupamento mais grosso)",
    "validacao": "leave-one-group-out sobre as configuracoes de treino",
    "n_ensaios": int(len(ensaios)),
    "n_configuracoes_perfil": int(k_perfil.nunique()),
    "n_configuracoes_13cat": int(k_13cat.nunique()),
    "run_idx_treino": [int(i) for i in idx_treino],
    "run_idx_teste": [int(i) for i in idx_teste],
    "n_dobras": int(len(dobras)),
    "tipos_sem_representacao_no_teste": sem_teste,
    # A atribuicao de dobra vai tambem para o JSON porque o .gitignore do projeto
    # ignora todo arquivo .csv, de modo que apenas o JSON pode ser versionado.
    "dobra_por_run_idx": {int(r.run_idx): (None if pd.isna(r.dobra) else int(r.dobra))
                          for r in particao.itertuples()},
    "run_case_por_run_idx": {int(r.run_idx): r.run_case for r in particao.itertuples()},
}
# Trava contra regeneracao acidental da particao.
#
# Este arquivo e a ancora do projeto: todo numero medido desde a terceira sprint
# pressupoe exatamente esta divisao de ensaios e esta atribuicao de dobras.
# Reexecutar o caderno sobre dados cuja chave mudou produziria uma particao
# diferente e gravaria por cima, sem erro algum, invalidando em silencio toda
# comparacao anterior. O risco nao e hipotetico: a chave e construida sobre as
# colunas de prefixo cfg_, que na quarta sprint deram lugar a prefixo reason_.
#
# A regeneracao continua possivel, mas deixa de ser acidental: basta apagar o
# arquivo antes de executar.
CAMPOS_ANCORA = ("run_idx_treino", "run_idx_teste", "dobra_por_run_idx")
if os.path.exists(SAIDA_JSON):
    with open(SAIDA_JSON, encoding="utf-8") as f:
        gravado = json.load(f)
    def _normaliza(v):
        """O JSON converte chave de dicionario em texto ao gravar, e o recalculo
        as produz como inteiro. Comparar sem normalizar acusaria divergencia em
        toda execucao, e a trava passaria a ser ignorada por quem a visse falhar
        sempre, que e o modo mais comum de uma verificacao morrer."""
        if isinstance(v, dict):
            return {str(k): v[k] for k in sorted(v, key=str)}
        return v

    divergem = [c for c in CAMPOS_ANCORA
                if json.dumps(_normaliza(gravado.get(c)), sort_keys=True)
                != json.dumps(_normaliza(registro[c]), sort_keys=True)]
    if divergem:
        raise AssertionError(
            "A particao recalculada diverge da gravada nos campos " + ", ".join(divergem)
            + ". Nada foi gravado. Se a intencao e mesmo redefinir a particao, apague "
            + os.path.abspath(SAIDA_JSON) + " e execute de novo, ciente de que todos os "
            "resultados anteriores do projeto deixam de ser comparaveis.")
    particao.to_csv(SAIDA_CSV, index=False)
    print("Particao recalculada coincide com a gravada; o JSON nao foi reescrito.")
    print(" ", os.path.abspath(SAIDA_JSON))
else:
    with open(SAIDA_JSON, "w", encoding="utf-8") as f:
        json.dump(registro, f, ensure_ascii=False, indent=2)
    particao.to_csv(SAIDA_CSV, index=False)
    print("Particao gravada pela primeira vez:")
    print(" ", os.path.abspath(SAIDA_CSV))
    print(" ", os.path.abspath(SAIDA_JSON))
print()
particao.head(10)

## 8. Como os experimentos devem consumir a partição

Estão previstos experimentos que combinam três fatores: as sete áreas térmicas, as duas versões de tratamento das perdas e os dois tratamentos de valores ausentes. As funções abaixo padronizam o consumo da partição, de modo que os resultados sejam comparáveis entre todas as combinações.

O tratamento de valores ausentes por imputação segue as regras definidas na revisão do notebook de nulos, avaliadas nesta ordem para cada coluna com ausências:

| Ordem | Condição | Ação |
|---|---|---|
| 1 | mais de 20 por cento de nulos e faixa superior a 35 graus Celsius | excluir a coluna |
| 2 | possui valor além de 2,5 vezes o intervalo interquartil | preencher com a mediana |
| 3 | nenhum valor extremo | preencher com a média |

A faixa é medida sem os valores extremos, entre o menor e o maior ponto dentro da cerca de 1,5 vez o intervalo interquartil, de modo que um único ponto distante não infle a amplitude e provoque a exclusão da coluna por si só. A escolha entre média e mediana acompanha a assimetria da distribuição, uma vez que a média é deslocada por valores extremos e a mediana não.

Três regras de método precisam ser observadas, sob pena de o resultado da comparação ser inválido.

A primeira diz respeito ao ajuste da imputação. Tanto o valor de preenchimento quanto a decisão de excluir são determinados exclusivamente sobre o conjunto de treino. Os arquivos de sufixo snull, gerados pelo notebook de tratamento de nulos, aplicam as mesmas regras sobre a base completa, o que faz o conjunto de teste contribuir para o valor imputado. A diferença entre as duas formas foi medida nesta base: o valor de preenchimento difere em 0,53 grau Celsius na média e em 3,13 graus no pior caso, e seis das 38 colunas trocam de regra entre média e mediana, enquanto a decisão de exclusão permanece a mesma. A magnitude é modesta, mas o ajuste sobre o treino é adotado por ser o procedimento correto e por não custar nada.

A segunda diz respeito à avaliação. As métricas devem ser calculadas somente sobre linhas cujo valor foi efetivamente medido, em qualquer das duas versões. Avaliar um modelo contra rótulos que a própria equipe fabricou mede a capacidade de reproduzir a imputação, e não a de prever a temperatura. A imputação, quando aplicada, restringe-se ao conjunto de treino.

A terceira diz respeito a qual conjunto sustenta a comparação. Os experimentos que confrontam as duas versões de tratamento das perdas e os dois tratamentos de valores ausentes são decisões de modelagem e, como tais, devem ser resolvidos sobre a validação cruzada, e não sobre o conjunto de teste. O teste é tocado uma única vez, ao final, para reportar o desempenho da configuração escolhida. Utilizá-lo na comparação teria dois efeitos indesejados: o número reportado deixaria de ser uma estimativa honesta, por ter influenciado a escolha, e a comparação se apoiaria em poucos ensaios, ao passo que a validação por leave-one-group-out percorre todas as configurações de treino.

In [ ]:
# Limiares das regras de tratamento de nulos, identicos aos do notebook revisado
PCT_NULOS_MAX = 20.0     # acima disso a coluna e candidata a exclusao
FAIXA_MAX = 35.0         # em graus Celsius, medida sem os valores extremos
MULT_EXTREMO = 2.5       # multiplo do IQR que define valor extremo


def perfila_coluna(serie):
    """Estatisticas de uma coluna numerica, ignorando os nulos."""
    s = serie.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1

    # cerca padrao de Tukey: separa o corpo dos dados dos pontos fora
    dentro = s[(s >= q1 - 1.5 * iqr) & (s <= q3 + 1.5 * iqr)]
    faixa = dentro.max() - dentro.min() if len(dentro) else 0.0

    extremos = int(((s <= q1 - MULT_EXTREMO * iqr) |
                    (s >= q3 + MULT_EXTREMO * iqr)).sum())

    n_nulos = int(serie.isna().sum())
    return {
        "nulos": n_nulos,
        "pct_nulos": 100 * n_nulos / len(serie),
        "IQR": iqr,
        "faixa": faixa,
        "extremos": extremos,
        "media": s.mean(),
        "mediana": s.median(),
    }


def decide_acao(perfil):
    """Aplica as tres regras, na ordem, e devolve (acao, motivo)."""
    if perfil["pct_nulos"] > PCT_NULOS_MAX and perfil["faixa"] > FAIXA_MAX:
        return "excluir", (f"{perfil['pct_nulos']:.1f}% de nulos e faixa "
                           f"{perfil['faixa']:.2f} C")
    if perfil["extremos"] > 0:
        return "mediana", f"{perfil['extremos']} valor(es) alem de {MULT_EXTREMO}x o IQR"
    return "media", "sem valores extremos"


def carrega_particao(arquivo=SAIDA_CSV):
    """Le a particao gravada. Todos os notebooks de treino devem usar esta funcao."""
    return pd.read_csv(resolve("preparacao_testes", arquivo))


def plano_de_nulos(df, indices_treino):
    """Acao decidida para cada coluna com nulos, usando SO o conjunto de treino."""
    entrada = set(colunas_de_configuracao(df))
    linhas = []
    for col in df.columns:
        if col in entrada or col == "run_idx":
            continue
        if not df[col].isna().any() or not pd.api.types.is_numeric_dtype(df[col]):
            continue
        perfil_col = perfila_coluna(df[col].iloc[indices_treino])
        acao, motivo = decide_acao(perfil_col)
        linhas.append({"coluna": col, "nulos": perfil_col["nulos"],
                       "% nulos": round(perfil_col["pct_nulos"], 1),
                       "faixa": round(perfil_col["faixa"], 2),
                       "extremos": perfil_col["extremos"], "acao": acao,
                       "valor": (None if acao == "excluir"
                                 else round(perfil_col[acao], 2)),
                       "motivo": motivo})
    return pd.DataFrame(linhas)


TRATAMENTOS = ("mascara", "imputado_treino", "imputado_base")


def carrega_area(area, versao="13cat", tratamento="mascara"):
    """Le o dataset de uma area sob o tratamento de nulos escolhido.

    mascara e imputado_treino partem do arquivo com os nulos preservados; o
    tratamento e aplicado adiante, em monta_xy, com estatistica do treino.
    imputado_base le o arquivo de sufixo snull, ja imputado sobre a base
    completa pelo notebook de tratamento de nulos.
    """
    if tratamento not in TRATAMENTOS:
        raise ValueError(f"tratamento deve ser um de {TRATAMENTOS}")

    if tratamento == "imputado_base":
        if versao != "13cat":
            raise FileNotFoundError(
                "o notebook de tratamento de nulos gera arquivos snull apenas para a "
                "versao 13cat; para a versao perfil use o tratamento imputado_treino"
            )
        return pd.read_csv(resolve(PASTA_13CAT, f"dataset_{area}_snull.csv"))

    return pd.read_csv(resolve(PASTA_13CAT, f"dataset_{area}_{versao}.csv"))


def monta_xy(df, alvo, indices, tratamento, indices_treino=None):
    """Monta X e y de um componente, restrito aos ensaios indicados.

    O tratamento e obrigatorio e nao possui valor padrao, de proposito: a escolha
    entre mascarar e imputar e um fator do experimento, e deixa-la implicita faria
    com que uma das alternativas fosse usada sem que ninguem tivesse decidido.

    Em mascara, apenas as linhas em que o componente foi medido sao usadas.
    Em imputado_treino, as regras revisadas sao aplicadas com estatistica calculada
    somente sobre o treino; uma coluna cuja acao seja excluir devolve (None, None).
    Em imputado_base, o arquivo lido ja vem preenchido e nada resta a fazer.

    O conjunto de avaliacao nunca deve receber valor imputado: use mascara para
    validacao e para teste, qualquer que seja o tratamento aplicado ao treino.
    """
    if tratamento not in TRATAMENTOS:
        raise ValueError(f"tratamento deve ser um de {TRATAMENTOS}")

    entrada = colunas_de_configuracao(df)
    sub = df.iloc[indices]

    if tratamento == "mascara":
        medido = sub[alvo].notna()
        return sub.loc[medido, entrada], sub.loc[medido, alvo]

    if tratamento == "imputado_base":
        return sub[entrada], sub[alvo]

    if indices_treino is None:
        raise ValueError("a imputacao exige os indices de treino, para evitar vazamento")

    perfil_col = perfila_coluna(df[alvo].iloc[indices_treino])
    acao, _ = decide_acao(perfil_col)
    if acao == "excluir":
        return None, None

    return sub[entrada], sub[alvo].fillna(perfil_col[acao])

### Demonstração

O plano de nulos é calculado sobre o conjunto de treino e comparado, em seguida, com o que resultaria da base completa. A comparação existe para tornar visível o efeito do vazamento que a segunda forma introduziria.

In [ ]:
p = carrega_particao()
tr = p.loc[p["conjunto"] == "treino", "run_idx"].tolist()
te = p.loc[p["conjunto"] == "teste", "run_idx"].tolist()

demo = pd.read_csv(resolve(PASTA_13CAT, "dataset_ipc_comp_13cat.csv"))

plano_treino = plano_de_nulos(demo, tr)
plano_base = plano_de_nulos(demo, list(range(len(demo))))

print("Plano de nulos do IPC Comp, ajustado sobre o treino:")
print(plano_treino.to_string(index=False))
print()

juncao = plano_treino.merge(plano_base, on="coluna", suffixes=("_treino", "_base"))
divergem = juncao[juncao["acao_treino"] != juncao["acao_base"]]
print(f"Colunas cuja acao mudaria se a estatistica viesse da base completa: {len(divergem)}")
if len(divergem):
    print(divergem[["coluna", "acao_treino", "valor_treino",
                    "acao_base", "valor_base"]].to_string(index=False))
print()

# Uso pratico, sobre a coluna de menor cobertura, nos tres tratamentos
alvo = "Heat Sink (Point/Probe)"
print(f"Alvo de demonstracao: {alvo}")
print()

for tratamento in TRATAMENTOS:
    try:
        dados = carrega_area("ipc_comp", versao="13cat", tratamento=tratamento)
        X, y = monta_xy(dados, alvo, tr, tratamento, indices_treino=tr)
    except FileNotFoundError as erro:
        print(f"  {tratamento:16s} indisponivel: {erro}")
        continue

    if X is None:
        print(f"  {tratamento:16s} coluna excluida pela regra de exclusao")
    else:
        print(f"  {tratamento:16s} treino {X.shape}, y sem nulos: {int(y.notna().sum())}")

# A avaliacao usa mascara em qualquer cenario, para nunca pontuar contra rotulo imputado
X_av, y_av = monta_xy(demo, alvo, te, "mascara")
print()
print(f"  avaliacao        {X_av.shape} (sempre mascarada, em todos os tratamentos)")

## 9. Resumo

A partição foi construída sobre a chave de configuração da versão com perfil de perdas, verificada como agrupamento mais grosso da versão com 13 categorias, e é portanto válida para as duas. O conjunto de teste foi sorteado por configuração, estratificado pelo tipo de ensaio, e a validação é feita por leave-one-group-out sobre as configurações de treino.

A partição foi gravada em arquivo e deve ser lida por todos os experimentos. Nenhum notebook de treino deve sortear a sua própria divisão, sob pena de as comparações entre áreas, entre versões de tratamento das perdas e entre tratamentos de valores ausentes ficarem confundidas com a variação do sorteio.